In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
train = pd.read_csv('battery_V5.csv')
test = pd.read_csv('test_battery_V5.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        task_type='GPU',   # GPU 사용 (Colab이면 가능)
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.5843512	test: 14.6399787	best: 14.6399787 (0)	total: 24.3ms	remaining: 24.3s
100:	learn: 13.4427838	test: 13.4918588	best: 13.4918588 (100)	total: 745ms	remaining: 6.63s
200:	learn: 12.5677587	test: 12.6106900	best: 12.6106900 (200)	total: 1.37s	remaining: 5.43s
300:	learn: 11.8946975	test: 11.9329325	best: 11.9329325 (300)	total: 1.99s	remaining: 4.62s
400:	learn: 11.3686213	test: 11.4003062	best: 11.4003062 (400)	total: 2.61s	remaining: 3.9s
500:	learn: 10.9501912	test: 10.9748238	best: 10.9748238 (500)	total: 4.88s	remaining: 4.86s
600:	learn: 10.6131200	test: 10.6312575	best: 10.6312575 (600)	total: 6.67s	remaining: 4.43s
700:	learn: 10.3446162	test: 10.3564838	best: 10.3564838 (700)	total: 7.29s	remaining: 3.11s
800:	learn: 10.1337837	test: 10.1413175	best: 10.1413175 (800)	total: 7.89s	remaining: 1.96s
900:	learn: 9.9689269	test: 9.9740500	best: 9.9740500 (900)	total: 8.5s	remaining: 934ms
999:	learn: 9.8422794	test: 9.8461831	best: 9.8461831 (999)	total: 9.11s	remai

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.6040400	test: 14.5606663	best: 14.5606663 (0)	total: 6.94ms	remaining: 6.93s
100:	learn: 13.4562888	test: 13.4288275	best: 13.4288275 (100)	total: 605ms	remaining: 5.39s
200:	learn: 12.5774912	test: 12.5649750	best: 12.5649750 (200)	total: 1.23s	remaining: 4.9s
300:	learn: 11.9012650	test: 11.9000587	best: 11.9000587 (300)	total: 1.82s	remaining: 4.23s
400:	learn: 11.3708200	test: 11.3795688	best: 11.3795688 (400)	total: 2.44s	remaining: 3.65s
500:	learn: 10.9474300	test: 10.9670862	best: 10.9670862 (500)	total: 3.04s	remaining: 3.03s
600:	learn: 10.6061387	test: 10.6351950	best: 10.6351950 (600)	total: 3.62s	remaining: 2.4s
700:	learn: 10.3342006	test: 10.3714088	best: 10.3714088 (700)	total: 4.22s	remaining: 1.8s
800:	learn: 10.1213463	test: 10.1642344	best: 10.1642344 (800)	total: 4.81s	remaining: 1.2s
900:	learn: 9.9553269	test: 10.0028731	best: 10.0028731 (900)	total: 6.41s	remaining: 704ms
999:	learn: 9.8284712	test: 9.8790713	best: 9.8790713 (999)	total: 8.55s	remai

...

0:	learn: 14.6107512	test: 14.5341487	best: 14.5341487 (0)	total: 7.01ms	remaining: 7s
100:	learn: 13.4686987	test: 13.3875975	best: 13.3875975 (100)	total: 683ms	remaining: 6.08s
200:	learn: 12.5938813	test: 12.5096062	best: 12.5096062 (200)	total: 1.28s	remaining: 5.1s
300:	learn: 11.9207037	test: 11.8322575	best: 11.8322575 (300)	total: 1.9s	remaining: 4.41s
400:	learn: 11.3919250	test: 11.3003700	best: 11.3003700 (400)	total: 2.5s	remaining: 3.74s
500:	learn: 10.9724125	test: 10.8782000	best: 10.8782000 (500)	total: 3.09s	remaining: 3.07s
600:	learn: 10.6335125	test: 10.5391062	best: 10.5391062 (600)	total: 3.7s	remaining: 2.46s
700:	learn: 10.3628269	test: 10.2699006	best: 10.2699006 (700)	total: 4.28s	remaining: 1.82s
800:	learn: 10.1501931	test: 10.0595794	best: 10.0595794 (800)	total: 4.89s	remaining: 1.22s
900:	learn: 9.9842500	test: 9.8953331	best: 9.8953331 (900)	total: 5.51s	remaining: 605ms
999:	learn: 9.8573100	test: 9.7694031	best: 9.7694031 (999)	total: 6.1s	remaining: 

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.5846750	test: 14.6382213	best: 14.6382213 (0)	total: 7.01ms	remaining: 7.01s
100:	learn: 13.4422800	test: 13.4972938	best: 13.4972938 (100)	total: 2.31s	remaining: 20.6s
200:	learn: 12.5673875	test: 12.6247225	best: 12.6247225 (200)	total: 4.38s	remaining: 17.4s
300:	learn: 11.8931912	test: 11.9535750	best: 11.9535750 (300)	total: 5s	remaining: 11.6s
400:	learn: 11.3626875	test: 11.4290300	best: 11.4290300 (400)	total: 5.59s	remaining: 8.35s
500:	learn: 10.9398875	test: 11.0128700	best: 11.0128700 (500)	total: 6.2s	remaining: 6.18s
600:	learn: 10.5988737	test: 10.6775087	best: 10.6775087 (600)	total: 6.82s	remaining: 4.53s
700:	learn: 10.3278963	test: 10.4106206	best: 10.4106206 (700)	total: 7.4s	remaining: 3.16s
800:	learn: 10.1153406	test: 10.1994044	best: 10.1994044 (800)	total: 8.02s	remaining: 1.99s
900:	learn: 9.9495975	test: 10.0344675	best: 10.0344675 (900)	total: 8.6s	remaining: 945ms
999:	learn: 9.8224913	test: 9.9087344	best: 9.9087344 (999)	total: 9.21s	remaini

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 14.5932038	test: 14.6043175	best: 14.6043175 (0)	total: 7.14ms	remaining: 7.14s
100:	learn: 13.4523450	test: 13.4628075	best: 13.4628075 (100)	total: 624ms	remaining: 5.55s
200:	learn: 12.5787250	test: 12.5871700	best: 12.5871700 (200)	total: 1.22s	remaining: 4.85s
300:	learn: 11.9049850	test: 11.9141863	best: 11.9141863 (300)	total: 1.86s	remaining: 4.33s
400:	learn: 11.3766200	test: 11.3844575	best: 11.3844575 (400)	total: 2.44s	remaining: 3.65s
500:	learn: 10.9563150	test: 10.9619725	best: 10.9619725 (500)	total: 4.85s	remaining: 4.83s
600:	learn: 10.6172700	test: 10.6204975	best: 10.6204975 (600)	total: 6.74s	remaining: 4.48s
700:	learn: 10.3463550	test: 10.3481963	best: 10.3481963 (700)	total: 7.35s	remaining: 3.13s
800:	learn: 10.1325931	test: 10.1364563	best: 10.1364563 (800)	total: 7.96s	remaining: 1.98s
900:	learn: 9.9658763	test: 9.9717619	best: 9.9717619 (900)	total: 8.55s	remaining: 939ms
999:	learn: 9.8381356	test: 9.8465656	best: 9.8465656 (999)	total: 9.16s	rem

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 9.8500


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V14.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
